# Interpretable Gearbox Fault Detection (KAN novelty) - Results AssemblyThis notebook assembles the curated outputs in `final_results/metrics/` and `final_results/plots/` into paper-ready figures.Novelty outputs include KAN feature-importance scores and pruning-derived surviving input features for each window size.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
METRICS = ROOT / 'metrics'

dl_accuracy = pd.read_csv(METRICS / 'dl_accuracy.csv', index_col=0)
dl_precision = pd.read_csv(METRICS / 'dl_precision.csv', index_col=0)
dl_recall = pd.read_csv(METRICS / 'dl_recall.csv', index_col=0)

baseline_accuracy = pd.read_csv(METRICS / 'baseline_accuracy.csv', index_col=0)
baseline_precision = pd.read_csv(METRICS / 'baseline_precision.csv', index_col=0)
baseline_recall = pd.read_csv(METRICS / 'baseline_recall.csv', index_col=0)

windows = [int(c) for c in baseline_accuracy.columns]
windows

In [ ]:
def baseline_best(df):
    # Best baseline model per window size
    return df.max(axis=0)

fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

ax[0].plot(windows, baseline_best(baseline_accuracy).values, marker='o', label='Baseline best')
ax[0].plot(windows, dl_accuracy.loc['KAN'].values, marker='o', label='KAN')
ax[0].plot(windows, dl_accuracy.loc['MLP'].values, marker='o', label='MLP')
ax[0].set_title('Accuracy (%)')
ax[0].set_xlabel('Window size (W)')
ax[0].set_ylabel('%')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(windows, baseline_best(baseline_precision).values, marker='o', label='Baseline best')
ax[1].plot(windows, dl_precision.loc['KAN'].values, marker='o', label='KAN')
ax[1].plot(windows, dl_precision.loc['MLP'].values, marker='o', label='MLP')
ax[1].set_title('Precision (%)')
ax[1].set_xlabel('Window size (W)')
ax[1].grid(True, alpha=0.3)

ax[2].plot(windows, baseline_best(baseline_recall).values, marker='o', label='Baseline best')
ax[2].plot(windows, dl_recall.loc['KAN'].values, marker='o', label='KAN')
ax[2].plot(windows, dl_recall.loc['MLP'].values, marker='o', label='MLP')
ax[2].set_title('Recall (%)')
ax[2].set_xlabel('Window size (W)')
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Novelty summary: pruning survival counts across windows
# (feature rankings are stored per-window in feature_importance_pruned_W{W}.csv)

survivor_counts = []
for W in windows:
    pruned_path = METRICS / f'pruned_survivors_W{W}.csv'
    if not pruned_path.exists():
        print(f'Missing: {pruned_path}')
        survivor_counts.append(None)
        continue
    pr = pd.read_csv(pruned_path)
    survivor_counts.append(len(pr))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(windows, survivor_counts, marker='o')
ax.set_title('KAN pruning: number of surviving input features')
ax.set_xlabel('Window size (W)')
ax.set_ylabel('Surviving features (count)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

pd.DataFrame({'W': windows, 'survivors_count': survivor_counts})

KAN network visualization PNGs are stored under `final_results/plots/kan_plots_W{W}/` and `final_results/plots/kan_plots_W{W}_pruned/`.